In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-23 00:30:50,949]::2549926179::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint is kept on disk.
# If a better validation MSE is found across any seed, this file is overwritten.
best_model_path = "./save_swintransformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(1):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        # Recreate DataLoaders for every seed so that shuffle
        # order is independently controlled.
        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # Fresh model / optimizer for each seed
        # ----------------------------------------------------
        model=ViViTBackbone(2, 14, 14, 2, 2, 2, 5, 96, 6, 8, 384, model=3)#num_frames=frame_nums,batchsz = batchsz
        model.to(device)
        criterion = torch.nn.MSELoss().to(device)  # 손실함수
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)
                audiodata = audiodata.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot, audiodata)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()
                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot, audiodata)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            current_val_mse = val_avg_loss.item()

            # ------------------------------------------------
            # Per-seed best + patience counter
            # ------------------------------------------------
            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds.
            # Only this ONE file is kept; it is overwritten.
            # ------------------------------------------------
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mse,
                }, best_model_path)

                logger.info(
                    "Global best updated | "
                    "seed: {} | epoch: {} | MSE: {:.4f}".format(
                        seed,
                        i + 1,
                        current_val_mse
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | MSE: {:.4f} | "
                "1-val_avg_o_loss: {:.4f} | "
                "1-val_avg_c_loss: {:.4f} | "
                "1-val_avg_e_loss: {:.4f} | "
                "1-val_avg_a_loss: {:.4f} | "
                "1-val_avg_n_loss: {:.4f}".format(
                    seed,
                    i + 1,
                    current_val_mse,
                    val_avg_o_loss.item(),
                    val_avg_c_loss.item(),
                    val_avg_e_loss.item(),
                    val_avg_a_loss.item(),
                    val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping for current seed
            # ------------------------------------------------
            print(
                f"Seed {seed} | Epoch {i+1} | Val MSE = {current_val_mse:.6f} | "
                f"Best MSE = {seed_best_mse:.6f} | Best Epoch = {seed_best_epoch} | "
                f"Patience = {no_improve_count}/{patience}"
            )

            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best MSE: {:.4f}".format(
                        seed,
                        i + 1,
                        seed_best_epoch,
                        seed_best_mse
                    )
                )
                break

        # ----------------------------------------------------
        # Keep scalar results only; do not retain seed models.
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mse": seed_best_mse,
            "best_1_mae": seed_best_mse
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MAE = {seed_best_mse:.6f} | "
            f"Best MSE = {seed_best_mse:.6f} | Patience = {patience}"
        )

        # ----------------------------------------------------
        # Release memory before next seed
        # ----------------------------------------------------
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary
# ============================================================

all_1_mae = np.array(
    [result["best_1_mae"] for result in seed_results],
    dtype=np.float64
)

mean_1_mae = all_1_mae.mean()
std_1_mae = all_1_mae.std(ddof=1) if len(all_1_mae) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | "
        "MAE {:.6f} | MSE {:.6f}".format(
            result["seed"],
            result["best_epoch"],
            result["best_mse"],
            result["best_1_mae"]
        )
    )

print("-" * 80)
print(f"Mean MSE : {mean_1_mae:.6f}")
print(f"Std  MSE : {std_1_mae:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"MSE={1.0 - global_best_mse:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 42 | Train 1/120: 100%|██████████| 1091/1091 [04:43<00:00,  3.84it/s]
Seed 42 | Valid 1/120: 100%|██████████| 191/191 [00:22<00:00,  8.35it/s]
[2026-09-23 00:36:06,802]::4118248011::INFO::Global best updated | seed: 42 | epoch: 1 | MSE: 1.5623
[2026-09-23 00:36:06,803]::4118248011::DEBUG::Seed: 42 | Epoch: 1 | MSE: 1.5623 | 1-val_avg_o_loss: 1.1226 | 1-val_avg_c_loss: 1.9502 | 1-val_avg_e_loss: 1.2344 | 1-val_avg_a_loss: 1.8825 | 1-val_avg_n_loss: 1.6216


Seed 42 | Epoch 1 | Val MSE = 1.562276 | Best MSE = 1.562276 | Best Epoch = 1 | Patience = 0/20


Seed 42 | Train 2/120: 100%|██████████| 1091/1091 [04:45<00:00,  3.82it/s]
Seed 42 | Valid 2/120: 100%|██████████| 191/191 [00:22<00:00,  8.63it/s]
[2026-09-23 00:41:17,079]::4118248011::INFO::Global best updated | seed: 42 | epoch: 2 | MSE: 1.4056
[2026-09-23 00:41:17,081]::4118248011::DEBUG::Seed: 42 | Epoch: 2 | MSE: 1.4056 | 1-val_avg_o_loss: 1.0695 | 1-val_avg_c_loss: 1.3397 | 1-val_avg_e_loss: 1.4783 | 1-val_avg_a_loss: 1.7003 | 1-val_avg_n_loss: 1.4402


Seed 42 | Epoch 2 | Val MSE = 1.405581 | Best MSE = 1.405581 | Best Epoch = 2 | Patience = 0/20


...

Seed 2024 | Epoch 27 | Val MSE = 1.378189 | Best MSE = 1.316766 | Best Epoch = 9 | Patience = 18/20


Seed 2024 | Train 28/120: 100%|██████████| 1091/1091 [04:42<00:00,  3.86it/s]
Seed 2024 | Valid 28/120: 100%|██████████| 191/191 [00:22<00:00,  8.60it/s]
[2026-09-23 13:41:52,357]::4118248011::DEBUG::Seed: 2024 | Epoch: 28 | MSE: 1.3844 | 1-val_avg_o_loss: 1.0785 | 1-val_avg_c_loss: 1.3974 | 1-val_avg_e_loss: 1.3873 | 1-val_avg_a_loss: 1.6096 | 1-val_avg_n_loss: 1.4495


Seed 2024 | Epoch 28 | Val MSE = 1.384449 | Best MSE = 1.316766 | Best Epoch = 9 | Patience = 19/20


Seed 2024 | Train 29/120: 100%|██████████| 1091/1091 [04:42<00:00,  3.86it/s]
Seed 2024 | Valid 29/120: 100%|██████████| 191/191 [00:22<00:00,  8.57it/s]
[2026-09-23 13:46:58,788]::4118248011::DEBUG::Seed: 2024 | Epoch: 29 | MSE: 1.3580 | 1-val_avg_o_loss: 1.0849 | 1-val_avg_c_loss: 1.3204 | 1-val_avg_e_loss: 1.3327 | 1-val_avg_a_loss: 1.6043 | 1-val_avg_n_loss: 1.4478
[2026-09-23 13:46:58,827]::4118248011::INFO::Early stopping | seed: 2024 | epoch: 29 | best epoch: 9 | best MSE: 1.3168


Seed 2024 | Epoch 29 | Val MSE = 1.358007 | Best MSE = 1.316766 | Best Epoch = 9 | Patience = 20/20
Seed 2024 finished | Best epoch = 9 | Best MAE = 1.316766 | Best MSE = 1.316766 | Patience = 20

MULTI-SEED RESULTS
Seed   42 | Best Epoch   4 | MAE 1.363494 | MSE 1.363494
Seed  123 | Best Epoch  33 | MAE 1.277739 | MSE 1.277739
Seed  777 | Best Epoch   7 | MAE 1.359312 | MSE 1.359312
Seed 1004 | Best Epoch   2 | MAE 1.315155 | MSE 1.315155
Seed 2024 | Best Epoch   9 | MAE 1.316766 | MSE 1.316766
--------------------------------------------------------------------------------
Mean MSE : 1.326493
Std  MSE : 0.035519
Global best: seed=123, epoch=33, MSE=-0.277739
Saved checkpoint: ./save_swintransformer_folder/model_best_multiseed.pth


In [12]:
# Explicit UDIVA MSE / patience summary
print("\n" + "=" * 80)
print("UDIVA MSE SUMMARY")
print("=" * 80)
print(f"Patience setting : {patience}")
print(f"Global Best MSE  : {global_best_mse:.6f}")
print(f"Global Best Seed : {global_best_seed}")
print(f"Best Epoch       : {global_best_epoch}")
print("=" * 80)



UDIVA MSE SUMMARY
Patience setting : 20
Global Best MSE  : 1.277739
Global Best Seed : 123
Best Epoch       : 33
